In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

DATA_DIR = Path("../data/raw/census_oa")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "TS001": "census2021-ts001-oa.csv",
    "TS002": "census2021-ts002-oa.csv",
    "TS003": "census2021-ts003-oa.csv",
    "TS004": "census2021-ts004-oa.csv",
    "TS007A": "census2021-ts007a-oa.csv",
    "TS016": "census2021-ts016-oa.csv",
    "TS021": "census2021-ts021-oa.csv",
    "TS044": "census2021-ts044-oa.csv",
    "TS054": "census2021-ts054-oa.csv",
    "TS063": "census2021-ts063-oa.csv",
    "TS066": "census2021-ts066-oa.csv",
    "TS067": "census2021-ts067-oa.csv",
}

In [8]:
def slugify(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"; measures: Value", "", text, flags=re.IGNORECASE)
    text = text.replace(":", " ")
    text = text.replace("(", " ").replace(")", " ")
    text = text.replace("/", " ")
    text = text.replace("&", " and ")
    text = re.sub(r"[^0-9a-zA-Z]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_").lower()


def clean_wide_census_file(path: Path, table_code: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)

    # Basic checks
    required = {"date", "geography", "geography code"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    # Rename identifiers
    df = df.rename(columns={
        "geography code": "OA21CD",
        "geography": "OA21NM_RAW",
        "date": "CENSUS_YEAR",
    })

    # Clean data columns
    rename_map = {}
    for col in df.columns:
        if col in ["OA21CD", "OA21NM_RAW", "CENSUS_YEAR"]:
            continue

        clean_col = slugify(f"{table_code}_{col}")
        rename_map[col] = clean_col

    df = df.rename(columns=rename_map)

    # Geography field is just the OA code in these files, not a useful name
    df["OA21CD"] = df["OA21CD"].astype(str).str.strip()
    df["OA21NM_RAW"] = df["OA21NM_RAW"].astype(str).str.strip()

    # Ensure count columns are numeric
    count_cols = [c for c in df.columns if c not in ["OA21CD", "OA21NM_RAW", "CENSUS_YEAR"]]
    for col in count_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

In [9]:
clean_tables = {}

for code, filename in FILES.items():
    path = DATA_DIR / filename
    df = clean_wide_census_file(path, code)

    clean_tables[code] = df

    print(code, filename)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Unique OAs:", df["OA21CD"].nunique())
    print(df.columns[:8].tolist())
    print("-" * 80)

TS001 census2021-ts001-oa.csv
Rows: 188880
Columns: 6
Unique OAs: 188880
['CENSUS_YEAR', 'OA21NM_RAW', 'OA21CD', 'ts001_residence_type_total', 'ts001_residence_type_lives_in_a_household', 'ts001_residence_type_lives_in_a_communal_establishment']
--------------------------------------------------------------------------------
TS002 census2021-ts002-oa.csv
Rows: 188880
Columns: 21
Unique OAs: 188880
['CENSUS_YEAR', 'OA21NM_RAW', 'OA21CD', 'ts002_marital_and_civil_partnership_status_total', 'ts002_marital_and_civil_partnership_status_never_married_and_never_registered_a_civil_partnership', 'ts002_marital_and_civil_partnership_status_married_or_in_a_registered_civil_partnership', 'ts002_marital_and_civil_partnership_status_married_or_in_a_registered_civil_partnership_married', 'ts002_marital_and_civil_partnership_status_married_or_in_a_registered_civil_partnership_married_opposite_sex']
--------------------------------------------------------------------------------
TS003 census2021-ts003-

In [10]:
oa_master = None

for code, df in clean_tables.items():
    if oa_master is None:
        oa_master = df.copy()
    else:
        # Keep only one copy of OA21NM_RAW and CENSUS_YEAR
        df_merge = df.drop(columns=["OA21NM_RAW", "CENSUS_YEAR"], errors="ignore")

        oa_master = oa_master.merge(
            df_merge,
            on="OA21CD",
            how="outer",
            validate="one_to_one"
        )

print("OA master shape:", oa_master.shape)
print("Unique OAs:", oa_master["OA21CD"].nunique())

OA master shape: (188880, 182)
Unique OAs: 188880


In [11]:
assert len(oa_master) == 188880
assert oa_master["OA21CD"].is_unique

In [12]:
oa_master.to_csv(OUT_DIR / "oa_counts_master_v1.csv", index=False)